In [0]:
Select * from workspace.default.resale_flat_prices limit 5;
Select * from workspace.default.rental_info limit 5;
Select * from workspace.default.property_info limit 5;

-- Create database namespaces
CREATE DATABASE IF NOT EXISTS bronze
  COMMENT 'Raw ingested data — never modified after load';

CREATE DATABASE IF NOT EXISTS silver
  COMMENT 'Cleaned, conformed, enriched';

CREATE DATABASE IF NOT EXISTS gold
  COMMENT 'Aggregated, business-ready for Power BI';


In [0]:
Select * from workspace.default.rental_info;

rent_approval_date,town,block,street_name,flat_type,monthly_rent
2021-01-01,ANG MO KIO,105,ANG MO KIO AVE 4,4-ROOM,2000
2021-01-01,ANG MO KIO,107,ANG MO KIO AVE 4,3-ROOM,1750
2021-01-01,ANG MO KIO,108,ANG MO KIO AVE 4,3-ROOM,1750
2021-01-01,ANG MO KIO,111,ANG MO KIO AVE 4,5-ROOM,2230
2021-01-01,ANG MO KIO,111,ANG MO KIO AVE 4,5-ROOM,2450
2021-01-01,ANG MO KIO,114,ANG MO KIO AVE 4,3-ROOM,1950
2021-01-01,ANG MO KIO,114,ANG MO KIO AVE 4,3-ROOM,1400
2021-01-01,ANG MO KIO,117,ANG MO KIO AVE 4,3-ROOM,1600
2021-01-01,ANG MO KIO,118,ANG MO KIO AVE 4,4-ROOM,2000
2021-01-01,ANG MO KIO,119,ANG MO KIO AVE 3,3-ROOM,1600


In [0]:
Select * from workspace.default.resale_flat_prices limit 5;

month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
2017-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
2017-01-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2017-01-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
2017-01-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
2017-01-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [0]:
Select * from workspace.default.property_info limit 5;

blk_no,street,max_floor_lvl,year_completed,residential,commercial,market_hawker,miscellaneous,multistorey_carpark,precinct_pavilion,bldg_contract_town,total_dwelling_units,1room_sold,2room_sold,3room_sold,4room_sold,5room_sold,exec_sold,multigen_sold,studio_apartment_sold,1room_rental,2room_rental,3room_rental,other_room_rental
1,BEACH RD,16,1970,Y,Y,N,N,N,N,KWN,142,0,1,138,1,2,0,0,0,0,0,0,0
1,BEDOK STH AVE 1,14,1975,Y,N,N,Y,N,N,BD,206,0,0,204,0,2,0,0,0,0,0,0,0
1,CANTONMENT RD,2,2010,N,Y,N,N,N,N,CT,0,0,0,0,0,0,0,0,0,0,0,0,0
1,CHAI CHEE RD,15,1982,Y,N,N,N,N,N,BD,102,0,0,0,10,92,0,0,0,0,0,0,0
1,CHANGI VILLAGE RD,4,1975,Y,Y,N,N,N,N,PRC,55,0,0,54,0,1,0,0,0,0,0,0,0


In [0]:
CREATE OR REPLACE TABLE bronze_
USING DELTA
AS
SELECT *
FROM workspace.default.resale_flat_prices;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bronze_property
USING DELTA
AS
SELECT *
FROM workspace.default.property_info;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bronze_rental
USING DELTA
AS
SELECT *
FROM workspace.default.rental_info;

num_affected_rows,num_inserted_rows


In [0]:
USE workspace.bronze;

In [0]:
CREATE OR REPLACE TABLE resale_bronze
USING DELTA
AS
SELECT *,
       current_timestamp() AS ingestion_time
FROM workspace.default.resale_flat_prices;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bronze_property
USING DELTA
AS
SELECT *,
       current_timestamp() AS ingestion_time
FROM workspace.default.property_info;

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE bronze_rental
USING DELTA
AS
SELECT *,
       current_timestamp() AS ingestion_time
FROM workspace.default.rental_info;

num_affected_rows,num_inserted_rows


In [0]:
SELECT DISTINCT r.block, r.street_name,r.town
FROM workspace.bronze.bronze_resale r
LEFT JOIN workspace.bronze.bronze_property p
ON r.block = p.blk_no
AND r.street_name = p.street
WHERE p.blk_no IS NULL;

block
82


In [0]:
SELECT DISTINCT r.block, r.street_name,r.town
FROM workspace.bronze.bronze_rental r
LEFT JOIN workspace.bronze.bronze_property p
ON r.block = p.blk_no
AND r.street_name = p.street
WHERE p.blk_no IS NULL;

block,street_name,town
519,WEST COAST RD,CLEMENTI
513,WEST COAST RD,CLEMENTI
26,TANGLIN HALT RD,QUEENSTOWN
25,TANGLIN HALT RD,QUEENSTOWN
28,TANGLIN HALT RD,QUEENSTOWN
40,TANGLIN HALT RD,QUEENSTOWN
30,TANGLIN HALT RD,QUEENSTOWN
27,TANGLIN HALT RD,QUEENSTOWN
63,C'WEALTH DR,QUEENSTOWN
82,MACPHERSON LANE,GEYLANG
